In [1]:
from models_style_steering import (
    collect_harmony_layer_states,
    tokenwise_knn_accuracy,
)
from generate_utils import load_SELaC, nucleus_token_by_token_generate, generate_files_with_nucleus
from train_utils import full_to_partial_masking
import torch
from torch.utils.data import DataLoader, ConcatDataset
import numpy as np
import pickle
from tqdm import tqdm
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset, CSGridMLM_collate_fn
import matplotlib.pyplot as plt
from copy import deepcopy
import os
from dotenv import load_dotenv

/home/maximos/miniconda3/envs/torch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

mask_token_id = tokenizer.mask_token_id
bar_token_id = tokenizer.bar_token_id

batchsize = 8

In [3]:
train_gjt = os.getenv('TRAIN_GJT')
train_nott = os.getenv('TRAIN_NOTT')

val_gjt = os.getenv('VAL_GJT')
val_nott = os.getenv('VAL_NOTT')

In [4]:
train_dataset_gjt = CSGridMLMDataset(train_gjt, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')
train_dataset_nott = CSGridMLMDataset(train_nott, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')

val_dataset_gjt = CSGridMLMDataset(val_gjt, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')
val_dataset_nott = CSGridMLMDataset(val_nott, tokenizer, frontloading=True, name_suffix='Q4_L80_bar_PC')


Loading data file.
Loading data file.
Loading data file.
Loading data file.


In [5]:
gjtTrainLoader = DataLoader(
    train_dataset_gjt,
    batch_size=batchsize,
    shuffle=False,
    drop_last=False,
    collate_fn=CSGridMLM_collate_fn
)

nottTrainLoader = DataLoader(
    train_dataset_nott,
    batch_size=batchsize,
    shuffle=False,
    drop_last=False,
    collate_fn=CSGridMLM_collate_fn
)

gjtValLoader = DataLoader(
    val_dataset_gjt,
    batch_size=batchsize,
    shuffle=False,
    drop_last=False,
    collate_fn=CSGridMLM_collate_fn
)

nottValLoader = DataLoader(
    val_dataset_nott,
    batch_size=batchsize,
    shuffle=False,
    drop_last=False,
    collate_fn=CSGridMLM_collate_fn
)

In [6]:
device_name = 'cuda:1'

if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

In [7]:
model = load_SELaC(
    tokenizer,
    device,
    guidance_dim=512,
    d_model=512,
    checkpoint_path=None,
)
model.eval()

SELaCModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder_layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayerWithAttn(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
      )
      (linear1): Linear(in_features=512, out_features=2048, bias=True)
      (dropout): Dropout(p=0.3, inplace=False)
      (linear2): Linear(in_features=2048, out_features=512, bias=True)
      (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.3, inplace=False)
      (dropout2): Dropout(p=0.3, inplace=False)
    )
  )
  (output_head): Linear(in_features=512, out_features=355, bias=True)
  (input_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (output_norm): LayerNorm((512,), 

In [8]:
# These must contain explicit mask-token IDs, preserving bar tokens.
nt_train_states, nt_train_harmony_ids, nt_train_valid = collect_harmony_layer_states(
    model, nottTrainLoader, mask_token_id, bar_token_id, device=device
)
jazz_train_states, jazz_train_harmony_ids, jazz_train_valid = collect_harmony_layer_states(
    model, gjtTrainLoader, mask_token_id, bar_token_id, device=device
)

nt_test_states, nt_test_harmony_ids, nt_test_valid = collect_harmony_layer_states(
    model, nottValLoader, mask_token_id, bar_token_id, device=device
)
jazz_test_states, jazz_test_harmony_ids, jazz_test_valid = collect_harmony_layer_states(
    model, gjtValLoader, mask_token_id, bar_token_id, device=device
)

In [9]:
# Combine the two training sets into a reference bank.
reference_states = {
    layer: torch.cat(
        [nt_train_states[layer], jazz_train_states[layer]], dim=0
    )
    for layer in nt_train_states
}

reference_labels = torch.cat([
    torch.zeros(len(train_dataset_nott), dtype=torch.long),
    torch.ones(len(train_dataset_gjt), dtype=torch.long),
])

reference_valid = torch.cat([nt_train_valid, jazz_train_valid])

query_states = {
    layer: torch.cat(
        [nt_test_states[layer], jazz_test_states[layer]], dim=0
    )
    for layer in nt_test_states
}

query_labels = torch.cat([
    torch.zeros(len(val_dataset_nott), dtype=torch.long),
    torch.ones(len(val_dataset_gjt), dtype=torch.long),
])

query_valid = torch.cat([nt_test_valid, jazz_test_valid])

In [10]:
results = tokenwise_knn_accuracy(
    query_states,
    reference_states,
    reference_labels,
    query_labels,
    k=5,
    # reference_valid=reference_valid,
    # query_valid=query_valid,
    reference_valid=None,
    query_valid=None,
    metric="cosine",
)

for layer, result in results.items():
    print(layer, result["token_accuracy"])

0 0.798972602739726
1 0.8412671232876713
2 0.872431506849315
3 0.8729452054794521
4 0.8799657534246575
5 0.910958904109589
6 0.9214041095890411
7 0.8702054794520548


In [11]:
nt_test_batch = next(iter(nottValLoader))

In [12]:
nt_test_melodies = nt_test_batch['pianoroll']
harmony_ids = nt_test_batch['harmony_ids']
harmony_input, _ = full_to_partial_masking(
    harmony_ids,
    mask_token_id,
    0,
    bar_token_id=bar_token_id
)
valid_positions = harmony_input[ harmony_input != mask_token_id ]

In [ ]:
# Start conservatively: move 5% toward the nearest Jazz state
# at every layer.
strengths = {
    layer: 0.9**(8-layer)
    for layer in range(len(model.encoder_layers))
}

model.eval()

with torch.no_grad():
    logits, steered_states, pre_out = model(
        nt_test_melodies[:1].to(device),
        harmony_tokens=harmony_input[:1].to(device),
        style_banks=jazz_train_states,
        steering_strengths=strengths,
        steer_mask=None,# valid_positions.to(device),
        get_layers_output=True,
    )

In [30]:
# print(logits.shape)
print(steered_states[0].shape)
print(pre_out[0].shape)

torch.Size([1, 80, 512])
torch.Size([1, 80, 512])


In [31]:
# input_f_path = '/media/maindisk/data/mel_harm_CA_all/gjt_CA_test/Autumn_Leaves.mxl'
input_f_path = '/media/maindisk/data/mel_harm_CA_all/nottingham_test/ashover12.mid'

piece_name = input_f_path.split('/')[-1].split('.')[0]
print(piece_name)

ashover12


In [32]:
gen_out = generate_files_with_nucleus(
    model,
    tokenizer,
    input_f_path=input_f_path,
    mxl_folder_out=None,
    midi_folder_out='midi_tests',
    name_suffix=f'{piece_name}_no',
    style_banks=jazz_train_states,
    steering_strengths=strengths,
    steer_mask=None,# valid_positions.to(device),
    use_constraints=False,
    intertwine_bar_info=True,
    normalize_tonality=True,
    temperature=1.0,
    p=0.9,
    unmasking_order='certain',
    create_gen=True,
    create_real=False
)
print(gen_out['gen_output_tokens'])

['<bar>', 'C:maj', 'C:maj', 'C:maj', 'C:maj7', '<bar>', 'C:maj', 'C:maj', 'C:maj', 'C:maj', '<bar>', 'C:7', 'G:7', 'G:7', 'G:7', '<bar>', 'C:maj', 'C:maj', 'G:7', 'G:7', '<bar>', 'F:maj', 'F:maj', 'C:maj', 'C:7', '<bar>', 'C:maj', 'C:maj', 'F:maj', 'C:7', '<bar>', 'C:7', 'F:maj', 'C:7', 'C:7', '<bar>', 'C:maj', 'C:maj', 'G:7', 'G:7', '<bar>', 'F:maj', 'F:maj', 'G:7', 'G:7', '<bar>', 'C:maj', 'C:maj', 'C:maj', 'C:maj', '<bar>', 'A:7', 'F:maj', 'B:7(b9)', 'B:7(b9)', '<bar>', 'G:min7', 'A:min7', 'C:maj7', 'C:maj7', '<bar>', 'D:min7', 'D:min7', 'C:maj', 'C:maj', '<bar>', 'C:maj', 'C:maj', 'C:maj', 'C:maj', '<bar>', 'F:maj', 'F:maj', 'G:7', 'G:7', '<bar>', 'C:maj7', 'C:maj7', 'C:maj7', 'C:maj7']
